In [1]:
# %% 셀 1: 데이터 로드 (Model 1 - Merged layout)
import os, json, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
GRID_W = 80
GRID_H = 80
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10
MAX_FRAMES = 2000
MAX_TEXT_LEN = 200
SCALAR_DIM = 9
TIME_DIM = 3

text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None
    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])
        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f
    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))
    video_name = data.get("video", "")
    file_id = os.path.basename(path)[:-5]
    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))
        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx, "y": gy, "w": gw, "h": gh,
        })
    stt_path = os.path.join(STT_DIR, channel, file_id + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass
    return {
        "channel": channel,
        "video_name": video_name,
        "file_id": file_id,
        "instances": inst_list,
        "stt_segments": stt_segments,
        "duration": duration,
    }


json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

samples = []
channel_set = set()
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        channel_set.add(result["channel"])
        samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

rng = random.Random(SEED)
by_channel = {}
for s in samples:
    by_channel.setdefault(s["channel"], []).append(s)

train_samples = []
eval_samples = []
for ch, ch_samples in by_channel.items():
    ch_samples.sort(key=lambda s: s["file_id"])
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

train_samples = [s for s in train_samples if len(s["instances"]) > 0]
eval_samples_with = [s for s in eval_samples if len(s["instances"]) > 0]

print(f"✅ 채널: {len(channels)}개")
print(f"   train: {len(train_samples):,}  eval: {len(eval_samples_with):,}")

✅ 임베딩 로드: 2,167,019개  |  dim=1024


로드: 100%|██████████| 66400/66400 [00:10<00:00, 6617.76it/s]


✅ 채널: 664개
   train: 56,892  eval: 2,984


In [2]:
# %% 셀 1b: 채널 서브샘플링 (debug)
rng_ch = random.Random(42)
sampled_channels = rng_ch.sample(channels, 67)
sampled_set = set(sampled_channels)

train_samples = [s for s in train_samples if s["channel"] in sampled_set]
eval_samples = [s for s in eval_samples if s["channel"] in sampled_set]
channels = sampled_channels
channel2id = {ch: i for i, ch in enumerate(channels)}

eval_samples_with = [s for s in eval_samples if len(s["instances"]) > 0]
train_samples = [s for s in train_samples if len(s["instances"]) > 0]

print(f"\n🔬 샘플링: {len(channels)}개 채널")
print(f"   train: {len(train_samples):,}  |  eval: {len(eval_samples):,}")


🔬 샘플링: 67개 채널
   train: 5,574  |  eval: 335


In [3]:
# %% 셀 2: Dataset + DataLoader (merged mask GT 생성)
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 8
NUM_WORKERS_DL = 8


class MergedLayoutDataset(Dataset):
    def __init__(self, samples, channel2id, text2emb):
        self.samples = [s for s in samples if len(s["instances"]) > 0]
        self.channel2id = channel2id
        self.text2emb = text2emb

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        channel_id = self.channel2id[s["channel"]]
        instances = s["instances"]
        duration = max(s["duration"], 0.1)
        n_frames = max(1, min(int(duration * FPS) + 1, MAX_FRAMES))
        n_inst = len(instances)

        inst_starts = np.array([inst["start"] for inst in instances])
        inst_ends   = np.array([inst["end"]   for inst in instances])
        inst_tlens  = np.array([inst["text_len"] for inst in instances])
        inst_cx = np.array([inst["x"] for inst in instances])
        inst_cy = np.array([inst["y"] for inst in instances])
        inst_w  = np.array([inst["w"] for inst in instances])
        inst_h  = np.array([inst["h"] for inst in instances])

        inst_x0 = np.maximum(0, inst_cx - inst_w // 2)
        inst_y0 = np.maximum(0, inst_cy - inst_h // 2)
        inst_x1 = np.minimum(GRID_W, inst_x0 + inst_w)
        inst_y1 = np.minimum(GRID_H, inst_y0 + inst_h)

        # 임베딩
        channel_emb = F.normalize(self.text2emb.get(s["channel"], ZERO_EMB), dim=-1)
        video_emb   = F.normalize(self.text2emb.get(s["video_name"], ZERO_EMB), dim=-1)
        inst_embs   = torch.stack([
            self.text2emb.get(inst["text"], ZERO_EMB) for inst in instances])
        inst_embs = F.normalize(inst_embs, dim=-1)

        inst_diff_ch  = inst_embs - channel_emb.unsqueeze(0)
        inst_diff_vid = inst_embs - video_emb.unsqueeze(0)
        inst_diff_stt = torch.zeros(n_inst, EMB_DIM)

        ch_sims  = F.cosine_similarity(inst_embs, channel_emb.unsqueeze(0), dim=-1).numpy()
        vid_sims = F.cosine_similarity(inst_embs, video_emb.unsqueeze(0), dim=-1).numpy()

        stt_sims = np.zeros(n_inst, dtype=np.float32)
        has_stts = np.zeros(n_inst, dtype=np.float32)
        stt_segments = s["stt_segments"]
        if len(stt_segments) > 0:
            stt_starts = np.array([seg["start"] for seg in stt_segments])
            stt_ends   = np.array([seg["end"]   for seg in stt_segments])
            stt_embs_raw = torch.stack([
                self.text2emb.get(seg["text"], ZERO_EMB) for seg in stt_segments])
            stt_embs = F.normalize(stt_embs_raw, dim=-1)
            for i in range(n_inst):
                mid = (inst_starts[i] + inst_ends[i]) / 2
                stt_active = (stt_starts <= mid) & (stt_ends > mid)
                stt_active_idx = np.where(stt_active)[0]
                if len(stt_active_idx) > 0:
                    inst_diff_stt[i] = inst_embs[i] - stt_embs[stt_active_idx[0]]
                    stt_sims[i] = F.cosine_similarity(
                        inst_embs[i].unsqueeze(0),
                        stt_embs[stt_active_idx[0]].unsqueeze(0)).item()
                    has_stts[i] = 1.0

        # 프레임별 활성 매트릭스
        times = np.arange(n_frames, dtype=np.float32) / FPS
        active_matrix = (
            (inst_starts[None, :] <= times[:, None] + 0.05) &
            (inst_ends[None, :]   >  times[:, None]))   # (T, n_inst)

        # avg_coactive
        co_active_per_frame = active_matrix.sum(axis=1)
        inst_avg_coactive = np.zeros(n_inst, dtype=np.float32)
        for i in range(n_inst):
            frames_i = active_matrix[:, i]
            if frames_i.any():
                inst_avg_coactive[i] = co_active_per_frame[frames_i].mean()
        inst_avg_coactive = np.log1p(inst_avg_coactive) / np.log1p(20.0)

        # inst_scalars 9d
        inst_scalars = np.zeros((n_inst, SCALAR_DIM), dtype=np.float32)
        inst_scalars[:, 0] = inst_tlens / MAX_TEXT_LEN
        inst_scalars[:, 1] = ch_sims
        inst_scalars[:, 2] = vid_sims
        inst_scalars[:, 3] = stt_sims
        inst_scalars[:, 4] = has_stts
        inst_scalars[:, 5] = inst_starts / max(duration, 0.1)
        inst_scalars[:, 6] = inst_ends / max(duration, 0.1)
        inst_scalars[:, 7] = (inst_ends - inst_starts) / max(duration, 0.1)
        inst_scalars[:, 8] = inst_avg_coactive

        # Merged mask per frame: 모든 active inst bbox의 union
        merged_mask = np.zeros((n_frames, GRID_H, GRID_W), dtype=np.float32)
        for j in range(n_inst):
            x0, y0, x1, y1 = int(inst_x0[j]), int(inst_y0[j]), int(inst_x1[j]), int(inst_y1[j])
            if x1 <= x0 or y1 <= y0:
                continue
            active_fr = np.where(active_matrix[:, j])[0]
            if len(active_fr) > 0:
                merged_mask[active_fr[:, None, None],
                            np.arange(y0, y1)[None, :, None],
                            np.arange(x0, x1)[None, None, :]] = 1.0

        # time feats
        time_feats = np.zeros((n_frames, TIME_DIM), dtype=np.float32)
        t_norm = times / max(duration, 1.0)
        time_feats[:, 0] = t_norm
        time_feats[:, 1] = np.sin(2 * np.pi * t_norm)
        time_feats[:, 2] = np.cos(2 * np.pi * t_norm)

        return {
            "channel_id":       torch.tensor(channel_id, dtype=torch.long),
            "inst_diff_ch":     inst_diff_ch,
            "inst_diff_vid":    inst_diff_vid,
            "inst_diff_stt":    inst_diff_stt,
            "inst_scalars":     torch.from_numpy(inst_scalars),
            "n_frames":         n_frames,
            "active_per_frame": torch.from_numpy(active_matrix),    # (T, n_inst)
            "merged_mask":      torch.from_numpy(merged_mask),       # (T, H, W)
            "time_feats":       torch.from_numpy(time_feats),
        }


def collate_fn(batch):
    max_T = max(b["n_frames"] for b in batch)
    max_inst = max(b["inst_diff_ch"].shape[0] for b in batch)
    B = len(batch)

    channel_ids   = torch.stack([b["channel_id"] for b in batch])
    inst_diff_ch  = torch.zeros(B, max_inst, EMB_DIM)
    inst_diff_vid = torch.zeros(B, max_inst, EMB_DIM)
    inst_diff_stt = torch.zeros(B, max_inst, EMB_DIM)
    inst_scalars  = torch.zeros(B, max_inst, SCALAR_DIM)
    active_per_frame = torch.zeros(B, max_T, max_inst, dtype=torch.bool)
    merged_mask = torch.zeros(B, max_T, GRID_H, GRID_W, dtype=torch.float32)
    time_feats = torch.zeros(B, max_T, TIME_DIM)
    frame_mask = torch.zeros(B, max_T, dtype=torch.bool)

    for i, b in enumerate(batch):
        T = b["n_frames"]
        n_inst = b["inst_diff_ch"].shape[0]
        inst_diff_ch[i, :n_inst]  = b["inst_diff_ch"]
        inst_diff_vid[i, :n_inst] = b["inst_diff_vid"]
        inst_diff_stt[i, :n_inst] = b["inst_diff_stt"]
        inst_scalars[i, :n_inst]  = b["inst_scalars"]
        active_per_frame[i, :T, :n_inst] = b["active_per_frame"]
        merged_mask[i, :T] = b["merged_mask"]
        time_feats[i, :T] = b["time_feats"]
        frame_mask[i, :T] = True

    return {
        "channel_ids":      channel_ids,
        "inst_diff_ch":     inst_diff_ch,
        "inst_diff_vid":    inst_diff_vid,
        "inst_diff_stt":    inst_diff_stt,
        "inst_scalars":     inst_scalars,
        "active_per_frame": active_per_frame,
        "merged_mask":      merged_mask,
        "time_feats":       time_feats,
        "frame_mask":       frame_mask,
    }


train_ds = MergedLayoutDataset(train_samples, channel2id, text2emb)
eval_ds  = MergedLayoutDataset(eval_samples, channel2id, text2emb)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS_DL, pin_memory=True,
    collate_fn=collate_fn, drop_last=True,
    persistent_workers=True,
)
eval_loader = DataLoader(
    eval_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS_DL, pin_memory=True,
    collate_fn=collate_fn,
    persistent_workers=True,
)

print(f"✅ Dataset: train={len(train_ds):,}  eval={len(eval_ds):,}")

batch = next(iter(train_loader))
print(f"\n✅ 배치 확인")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {tuple(v.shape)}  {v.dtype}")
print(f"\n  merged_mask positive ratio: {batch['merged_mask'].mean().item()*100:.2f}%")

✅ Dataset: train=5,574  eval=288

✅ 배치 확인
  channel_ids: (8,)  torch.int64
  inst_diff_ch: (8, 108, 1024)  torch.float32
  inst_diff_vid: (8, 108, 1024)  torch.float32
  inst_diff_stt: (8, 108, 1024)  torch.float32
  inst_scalars: (8, 108, 9)  torch.float32
  active_per_frame: (8, 956, 108)  torch.bool
  merged_mask: (8, 956, 80, 80)  torch.float32
  time_feats: (8, 956, 3)  torch.float32
  frame_mask: (8, 956)  torch.bool

  merged_mask positive ratio: 2.88%


In [4]:
# %% 셀 3: 모델 정의 (Merged layout: inst encoder + frame aggregator + spatial codebook decoder)
import math

D_MODEL = 256
D_PIX = 128
N_HEADS = 8
N_LAYERS_INST = 4
D_FF = 512
DROPOUT = 0.1


class MergedLayoutModel(nn.Module):
    def __init__(self, n_channels, emb_dim=EMB_DIM, d_model=D_MODEL, d_pix=D_PIX,
                 n_heads=N_HEADS, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.d_model = d_model
        self.d_pix = d_pix

        # ── 1. Instance projections (train_3와 동일 구조) ──
        self.ch_proj = nn.Sequential(nn.Linear(emb_dim + 1, d_model // 2), nn.GELU())
        self.vid_proj = nn.Sequential(nn.Linear(emb_dim + 1, d_model // 2), nn.GELU())
        self.stt_proj = nn.Sequential(nn.Linear(emb_dim + 2, d_model // 2), nn.GELU())
        self.len_proj = nn.Sequential(nn.Linear(5, d_model // 2), nn.GELU())
        self.slot_combine = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.GELU(),
            nn.Linear(d_model, d_model))

        inst_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu")
        self.inst_self_attn = nn.TransformerEncoder(
            inst_layer, num_layers=N_LAYERS_INST, enable_nested_tensor=False)

        self.channel_emb = nn.Embedding(n_channels, d_model)

        # ── 2. Frame aggregator ──
        self.time_proj = nn.Linear(TIME_DIM, d_model)
        self.count_proj = nn.Sequential(
            nn.Linear(1, d_model // 4), nn.GELU(),
            nn.Linear(d_model // 4, d_model))

        # ── 3. Heatmap decoder via spatial codebook ──
        self.frame_proj = nn.Sequential(
            nn.Linear(d_model, d_pix), nn.GELU(),
            nn.Linear(d_pix, d_pix))
        self.spatial_codebook = nn.Parameter(torch.randn(d_pix, GRID_H, GRID_W) * 0.02)
        self.heatmap_bias = nn.Parameter(torch.zeros(GRID_H, GRID_W))
        self.logit_scale = nn.Parameter(torch.tensor(1.0 / math.sqrt(d_pix)))

    def encode_inst(self, channel_ids, inst_diff_ch, inst_diff_vid,
                    inst_diff_stt, inst_scalars):
        ch_input = torch.cat([inst_diff_ch, inst_scalars[..., 1:2]], dim=-1)
        vid_input = torch.cat([inst_diff_vid, inst_scalars[..., 2:3]], dim=-1)
        stt_input = torch.cat([inst_diff_stt,
                               inst_scalars[..., 3:4],
                               inst_scalars[..., 4:5]], dim=-1)
        len_input = torch.cat([
            inst_scalars[..., 0:1],
            inst_scalars[..., 5:8],
            inst_scalars[..., 8:9]], dim=-1)

        proj_ch  = self.ch_proj(ch_input)
        proj_vid = self.vid_proj(vid_input)
        proj_stt = self.stt_proj(stt_input)
        proj_len = self.len_proj(len_input)

        inst_tokens = self.slot_combine(
            torch.cat([proj_ch, proj_vid, proj_stt, proj_len], dim=-1))
        ch_emb = self.channel_emb(channel_ids).unsqueeze(1)
        inst_tokens = inst_tokens + ch_emb

        inst_mask = (inst_scalars[..., 0] != 0)
        inst_pad = ~inst_mask
        inst_tokens = self.inst_self_attn(inst_tokens, src_key_padding_mask=inst_pad)
        return inst_tokens, inst_mask

    def aggregate_frames(self, inst_tokens, active_per_frame, time_feats, channel_ids):
        """
        inst_tokens: (B, N_inst, D)
        active_per_frame: (B, T, N_inst) bool
        time_feats: (B, T, TIME_DIM)
        Returns: frame_tokens (B, T, D)
        """
        active_f = active_per_frame.float()
        count = active_f.sum(dim=-1, keepdim=True)                          # (B, T, 1)
        weight = active_f / count.clamp(min=1.0)                             # (B, T, N_inst)
        frame_tokens = torch.einsum('btn,bnd->btd', weight, inst_tokens)    # (B, T, D)

        ch_emb = self.channel_emb(channel_ids).unsqueeze(1)                  # (B, 1, D)
        time_emb = self.time_proj(time_feats)                                # (B, T, D)
        count_norm = torch.log1p(count) / math.log1p(20.0)                   # (B, T, 1)
        count_emb = self.count_proj(count_norm)                              # (B, T, D)
        return frame_tokens + ch_emb + time_emb + count_emb

    def predict_heatmap(self, frame_tokens):
        """frame_tokens: (B, T, D) → heatmap_logits: (B, T, H, W)"""
        proj = self.frame_proj(frame_tokens)                                 # (B, T, D_pix)
        heatmap = torch.einsum('btd,dhw->bthw', proj, self.spatial_codebook)
        heatmap = heatmap * self.logit_scale + self.heatmap_bias
        return heatmap

    def forward(self, channel_ids, inst_diff_ch, inst_diff_vid, inst_diff_stt,
                inst_scalars, active_per_frame, time_feats, frame_mask):
        inst_tokens, _ = self.encode_inst(
            channel_ids, inst_diff_ch, inst_diff_vid, inst_diff_stt, inst_scalars)
        frame_tokens = self.aggregate_frames(
            inst_tokens, active_per_frame, time_feats, channel_ids)
        heatmap_logits = self.predict_heatmap(frame_tokens)
        return heatmap_logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MergedLayoutModel(n_channels=len(channels)).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🖥️  Device: {device}")
print(f"📐 파라미터: {n_params:,}")
print(f"   inst self-attn {N_LAYERS_INST}L (D={D_MODEL})")
print(f"   frame aggregator: mean-pool active inst + count + time + channel")
print(f"   spatial codebook ({D_PIX}×{GRID_H}×{GRID_W}) → 80×80 heatmap")

🖥️  Device: cuda
📐 파라미터: 3,610,369
   inst self-attn 4L (D=256)
   frame aggregator: mean-pool active inst + count + time + channel
   spatial codebook (128×80×80) → 80×80 heatmap


In [5]:
# %% 셀 4: 학습 (BCE + Dice on merged heatmap, best-threshold 평가)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 30
LR = 1e-4
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 1.0
DICE_WEIGHT = 1.0
SAVE_DIR = "./model/8_layout_1_merged"
os.makedirs(SAVE_DIR, exist_ok=True)

USE_BF16 = (device.type == "cuda")
EVAL_THRESHOLDS = [i / 100 for i in range(5, 96, 5)]   # 0.05 ~ 0.95


def merged_loss(logits, gt_mask, frame_mask):
    """BCE + Dice on merged heatmap.
    logits: (B, T, H, W)
    gt_mask: (B, T, H, W) float ∈ [0, 1]
    frame_mask: (B, T) bool
    """
    fm = frame_mask.unsqueeze(-1).unsqueeze(-1).float()         # (B, T, 1, 1)

    # BCE
    bce_per_pixel = F.binary_cross_entropy_with_logits(
        logits, gt_mask, reduction='none')
    bce = (bce_per_pixel * fm).sum() / (fm.sum() * GRID_H * GRID_W).clamp(min=1.0)

    # Dice (per frame, then averaged)
    pred = torch.sigmoid(logits.float())
    pred_m = pred * fm
    gt_m = gt_mask * fm
    inter = (pred_m * gt_m).sum(dim=(-1, -2))                    # (B, T)
    union = pred_m.sum(dim=(-1, -2)) + gt_m.sum(dim=(-1, -2))    # (B, T)
    dice = (2 * inter + 1.0) / (union + 1.0)
    dice_loss = ((1 - dice) * frame_mask.float()).sum() / frame_mask.sum().clamp(min=1.0)

    return bce + DICE_WEIGHT * dice_loss, {
        'bce': float(bce.detach()),
        'dice': float(dice_loss.detach()),
    }


@torch.no_grad()
def compute_metrics_best_th(logits, gt_mask, frame_mask, thresholds=EVAL_THRESHOLDS):
    """Best-threshold search by IoU. sigmoid은 한 번만, threshold만 sweep."""
    pred_prob = torch.sigmoid(logits.float())
    gt = (gt_mask > 0.5)
    fm = frame_mask.unsqueeze(-1).unsqueeze(-1)
    valid_gt = gt & fm
    n_gt_pos = valid_gt.sum().item()

    best = None
    for th in thresholds:
        valid_pred = (pred_prob > th) & fm
        tp = (valid_pred & valid_gt).sum().item()
        fp = (valid_pred & ~valid_gt).sum().item()
        fn = n_gt_pos - tp
        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        iou = tp / (tp + fp + fn + 1e-8)
        m = {'P': p, 'R': r, 'F1': f1, 'IoU': iou, 'th': th}
        if best is None or m['IoU'] > best['IoU']:
            best = m
    return best


def to_device(batch, dev):
    return {k: (v.to(dev, non_blocking=True) if isinstance(v, torch.Tensor) else v)
            for k, v in batch.items()}


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS * max(1, len(train_loader)))

best_eval = 0.0
log_path = os.path.join(SAVE_DIR, "train_log.txt")

for epoch in range(EPOCHS):
    # ── train ──
    model.train()
    tr_loss = 0.0
    tr_bce = 0.0
    tr_dice = 0.0
    tr_n = 0
    pbar = tqdm(train_loader, desc=f"E{epoch+1}/{EPOCHS} train")
    for batch in pbar:
        batch = to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=USE_BF16):
            logits = model(
                batch["channel_ids"],
                batch["inst_diff_ch"],
                batch["inst_diff_vid"],
                batch["inst_diff_stt"],
                batch["inst_scalars"],
                batch["active_per_frame"],
                batch["time_feats"],
                batch["frame_mask"])
            loss, stats = merged_loss(
                logits, batch["merged_mask"], batch["frame_mask"])

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        tr_loss += loss.item()
        tr_bce += stats['bce']
        tr_dice += stats['dice']
        tr_n += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}",
                         bce=f"{stats['bce']:.4f}",
                         dice=f"{stats['dice']:.4f}")

    # ── eval ──
    model.eval()
    ev_loss = 0.0
    ev_n = 0
    metrics_acc = {'P': 0.0, 'R': 0.0, 'F1': 0.0, 'IoU': 0.0}
    th_acc = 0.0
    with torch.no_grad():
        for batch in tqdm(eval_loader, desc=f"E{epoch+1}/{EPOCHS} eval"):
            batch = to_device(batch, device)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=USE_BF16):
                logits = model(
                    batch["channel_ids"],
                    batch["inst_diff_ch"],
                    batch["inst_diff_vid"],
                    batch["inst_diff_stt"],
                    batch["inst_scalars"],
                    batch["active_per_frame"],
                    batch["time_feats"],
                    batch["frame_mask"])
                loss, _ = merged_loss(logits, batch["merged_mask"], batch["frame_mask"])
            ev_loss += loss.item()
            m = compute_metrics_best_th(
                logits, batch["merged_mask"], batch["frame_mask"])
            for k in metrics_acc:
                metrics_acc[k] += m[k]
            th_acc += m['th']
            ev_n += 1

    tr_avg = tr_loss / max(tr_n, 1)
    ev_avg = ev_loss / max(ev_n, 1)
    for k in metrics_acc:
        metrics_acc[k] /= max(ev_n, 1)
    th_avg = th_acc / max(ev_n, 1)

    line = (f"E{epoch+1:02d}  train={tr_avg:.4f}  eval={ev_avg:.4f}  "
            f"mF1={metrics_acc['F1']:.4f}  IoU={metrics_acc['IoU']:.4f}  "
            f"P={metrics_acc['P']:.4f}  R={metrics_acc['R']:.4f}  "
            f"th={th_avg:.2f}  "
            f"bce={tr_bce/max(tr_n,1):.4f}  dice={tr_dice/max(tr_n,1):.4f}")
    print(line)
    with open(log_path, "a") as f:
        f.write(line + "\n")

    score = metrics_acc['IoU']
    if score > best_eval:
        best_eval = score
        torch.save({
            "model": model.state_dict(),
            "epoch": epoch + 1,
            "metrics": metrics_acc,
            "best_th": th_avg,
            "channel2id": channel2id,
        }, os.path.join(SAVE_DIR, "best.pt"))
        print(f"  ✅ best saved (IoU={score:.4f}, th={th_avg:.2f})")

    torch.save({"model": model.state_dict(), "epoch": epoch + 1},
               os.path.join(SAVE_DIR, "last.pt"))


E1/30 eval: 100%|██████████| 36/36 [00:04<00:00,  8.53it/s]


E01  train=1.0844  eval=1.0084  mF1=0.3696  IoU=0.2367  P=0.3129  R=0.5210  th=0.38  bce=0.3200  dice=0.7644
  ✅ best saved (IoU=0.2367, th=0.38)


E2/30 eval: 100%|██████████| 36/36 [00:02<00:00, 15.45it/s]


E02  train=0.9942  eval=1.0012  mF1=0.3780  IoU=0.2430  P=0.3095  R=0.5625  th=0.32  bce=0.2472  dice=0.7470
  ✅ best saved (IoU=0.2430, th=0.32)


E3/30 eval: 100%|██████████| 36/36 [00:02<00:00, 15.11it/s]


...

E28  train=0.5391  eval=0.5446  mF1=0.5787  IoU=0.4187  P=0.5667  R=0.6192  th=0.43  bce=0.1670  dice=0.3721
  ✅ best saved (IoU=0.4187, th=0.43)


E29/30 eval: 100%|██████████| 36/36 [00:02<00:00, 15.55it/s]


E29  train=0.5382  eval=0.5446  mF1=0.5786  IoU=0.4187  P=0.5656  R=0.6200  th=0.42  bce=0.1673  dice=0.3708


E30/30 eval: 100%|██████████| 36/36 [00:02<00:00, 15.56it/s]


E30  train=0.5380  eval=0.5446  mF1=0.5786  IoU=0.4186  P=0.5658  R=0.6198  th=0.42  bce=0.1668  dice=0.3711
